# Semana 4 · Diagnóstico y limpieza inicial de datos (Python)

**Práctica obligatoria · 2 pomodoros (40 minutos)**

En este notebook vas a diagnosticar problemas de calidad, trabajar sobre una copia de los datos y documentar cada decisión. No solo deberás escribir el código, sino también interpretar el problema y justificar el tratamiento.

> Realizá esta versión o la versión en R. No es necesario completar ambas.

**Resultados esperados**

Al finalizar deberías poder:

- revisar estructura, tipos, valores faltantes y duplicados;
- distinguir errores verificables de valores atípicos potencialmente válidos;
- registrar problemas, decisiones y justificaciones en una bitácora.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Práctica Resuelta

## Dataset

Trabajaremos con el dataset 'Titanic'. Este conjunto de datos registra información sobre los pasajeros del trágico viaje del RMS Titanic en 1912. Este dataset se usa para realizar el análisis de datos y estudiar las características comunes de los pasajeros que sobrevivieron al accidente. Pero tiene un problema, contiene una gran variedad de errores y problemas. Por lo tanto, hay que realizar la limpieza de datos.

Aquí se describen en detalle las variables del dataset:

| Variable      | Descripción                                                                                                        |
| :------------ | :----------------------------------------------------------------------------------------------------------------- |
| `PassengerId` | Identificador único del pasajero.                                                                                  |
| `Survived`    | Indica si el pasajero sobrevivió (1) o no (0).                                                                     |
| `Pclass`      | Clase del pasaje (1ª, 2ª o 3ª clase). Es un indicador del estatus socioeconómico.                                     |
| `Name`        | Nombre del pasajero.                                                                                               |
| `Sex`         | Género del pasajero (male/female).                                                                                 |
| `Age`         | Edad del pasajero en años. Los valores fraccionarios representan edades estimadas.                                   |
| `SibSp`       | Número de hermanos/cónyuges a bordo del Titanic.                                                                   |
| `Parch`       | Número de padres/hijos a bordo del Titanic.                                                                        |
| `Ticket`      | Número de ticket.                                                                                                  |
| `Fare`        | Tarifa pagada por el pasaje.                                                                                       |
| `Cabin`       | Número de cabina. Muchos valores faltantes.                                                                       |
| `Embarked`    | Puerto de embarque (C = Cherbourg, Q = Queenstown, S = Southampton).                                               |

Con la siguiente celda podrás obtener el dataset completo. Si llega a generar errores o problemas, podés descargar el dataset que está subido en el aula virtual, subirlo a la carpeta "Archivos" de colab y ejecutar el siguiente código:

```
df_titanic = pd.load_csv('\content\titanic.csv')
```


In [2]:
from pathlib import Path

DATA_DIR = Path('../datos')
df_titanic = pd.read_csv(DATA_DIR / 'titanic.csv')
df_titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## 2. Diagnóstico inicial

Antes de modificar, debemos observar la cantidad de registros que tienen, los tipos de variables (columnas) y un resumen estadístico sobre el comportamiento de cada una.

### Tamaño del dataset

In [3]:
print(df_titanic.shape)

(891, 12)


Posee 891 filas y 12 columnas

### Tipos de variables

In [4]:
df_titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Observamos que el dataset tiene 891 filas en total. Contiene 12 variables: 2 de tipo float (cuantitativas continuas), 5 de tipo int (cuantitativas, posiblemente discretas) y 5 de tipo object (cualitativas nomilase y ordinales).

También observamos la cantidad de valores no nulos que posee cada variable. Fácilmente detectamos varios valores nulos que posteriormente deberemos corregir.

### Resumen estadístico

In [5]:
df_titanic.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
df_titanic.describe(include=['object','category'])

C:\Users\Pollo\AppData\Local\Temp\ipykernel_15344\1487857994.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_titanic.describe(include=['object','category'])


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


De aquí podemos obtener distintas conclusiones:
* La variable `age` va de 0.42 hasta 80. La media es 29.7 y la mediana 28, lo que indica una leve asimetría hacia la derecha. También podemos decir que el caso del 0.42 refiere a un bebé de menos de un año.
* La variable `SibSp` tiene un cuartil 3 igual a 0, lo que indica que la gran mayoría de los pasajeros viajaron sin hermanos ni cónyuges. Sin embargo, existen pocos casos que sí han viajado con hermanos/cónyuges, incluso llegando a una cantidad máxima de 8. Algo similar ocurre con `Parch`.
* Las variables `sex` y `Embarked` son las varibles cualitativas que contienen una cantidad coherente de valores únicos. Las otras variables deberán ser examinadas para evaluar la información que nos puede aportar.

Pensá que otra conclusión se puede extraer de este resultado.

## Limpieza de datos

Ahora vamos a realizar la limpieza de los datos para limpiar todos los errores e imperfecciones que contenga el dataset.

### 0- Preparativos

Primero vamos a hacer una copia del dataset original para dejarlo como respaldo. Es una buena practica cuando se está modificando los datos. En caso de cometer algún error o inconveniente, se puede volver al dataset original y recuperar estos datos.

In [7]:
df_titanic_original = df_titanic.copy()

También, crearemos un documento que llamaremos "Bitácora" para dejar constancia del problema detectado, las variables afectadas, la decisión tomada y la justificación de dicha decisión.

In [8]:
registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        "problema_detectado": problema,
        "variable_afectada": variable,
        "decision_tomada": decision,
        "justificacion": justificacion
    })

### 1- Detección de valores duplicados

Siempre hay que revisar que no existan registros duplicados dentro del dataset. En caso de que existan, entonces habrá que eliminarlos para que evitar tener datos sesgados. Pero cuidado, registros duplicados no siempre significa mediciones registradas varias veces; puede ocurrir que -tomando este contexto- dos personas diferentes hayan conseguido la misma clase al misma precio y hyan sido acompañado por la misma cantidad de hermanos, padres e hijos. En estos casos, hay que mantener esta duplicación de filas ya que no se trata de registros registrados dos veces sino de dos individuos con las mismas características.

En nuestro caso, como tenemos la columna PassengerId, consideraremos registros duplicados facilmente si todas las columnas son iguales.

In [9]:
df_titanic.duplicated().sum()

np.int64(0)

observamos que no existen valores duplicados. Por lo tanto, no necesitamos realizar ninguna acción en este paso y contiuamos con el resto de la limpieza de datos.

### 2- Selección de variables

Como primer paso, volveremos a ver el dataframe y evaluaremos si existen columnas que no aporten información a la hora de realizar el análisis de los datos.

In [10]:
df_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Realicemos el análisis de la variable `PassengerId`. Parece representar un identificador de los pasajeros, pero pasemos a corroborar esta suposición.

In [11]:
# Observo la cantidad de valores únicos que contiene la variable PassengerId
len(df_titanic['PassengerId'].unique())

891

Esta variable contiene 891 valores únicos. Pero el dataset recibido tiene 891 registros, por lo tanto esta variable es un identificador con valores únicos que no tienen ninguna relación con las demás variables. En conclusión, esta variable debe ser eliminada.

Lo mismo ocurre con la columna `Name`, ya que el nombre no suele ser un valor importante al momento de analizar

Por último, analizando la variable `Ticket` vemos que cae exactamente en el mismo inconveniente. Es un identificador que no aporta información.

In [12]:
# Tomamos una muestra aleatoria de los tickets
df_titanic['Ticket'].sample(20)

126              370372
423              347080
381                2653
729    STON/O2. 3101271
342              248740
299            PC 17558
735               54636
802              113760
653              330919
757               29108
476               31027
680              330935
810                3474
743              376566
809              113806
536              113050
742            PC 17608
344              229236
326              345364
507              111427
Name: Ticket, dtype: str

En definitiva, se deberán eliminar las variables `PassengerId`, `Name` y `Ticket`

In [13]:
# Elimino las columnas no deseadas. El parámetro inplace=True permite sobrescribir los cambios sobre el mismo dataframe
df_titanic.drop(columns=['PassengerId','Name','Ticket'], inplace=True)

Registramos el cambio en la bitácora

In [14]:
registrar("Selección de variables para el análisis",
          "PassengerId-Name-Ticket",
          "Eliminación de las columnas",
          "Estas columnas contienen nombres y códigos de identificación, por lo que no aportan ningún tipo de información")

### 3- Tratamiento de valores faltantes o nulos

Ahora veamos cuantos valores nulos existen en este dataset

In [15]:
df_titanic.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64

Observamos que 3 variables tienen valores nulos: `Age`, `Cabin` y `Embarked`

En el caso de `Cabin` vemos que tiene una cantidad muy elevada de valores nulos. Podemos hacer un pequeño cálculo para ver la proporción de valores faltantes.

In [16]:
proporcion_nulos = df_titanic['Cabin'].isna().sum()/df_titanic.shape[0]
print(f'La proporción de valores nulos es de {round(proporcion_nulos*100,2)}%')

La proporción de valores nulos es de 77.1%


La columna `Cabin` tiene ¡77% de valores faltantes! Es decir, casi 4 de cada 5 filas no contiene valores. Como la tasa de pérdida de información es elevada y no se observa una relación coherente sobre esta pérdida de valores, entonces eliminamos la columna completa del análisis.

In [17]:
df_titanic.drop(columns=['Cabin'], inplace=True)

# Validamos que se haya realizado correctamente la modificación
df_titanic.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [18]:
registrar("Tratamiento de valores faltantes",
          "Cabin",
          "Eliminación de la columna",
          "La columna contiene 77% de valores faltantes, por lo que no aporta datos suficientes para obtener información")

Ahora veamos la variable `Embarked`. Ésta tiene 2 valores nulos, una cantidad muy baja. Podríamos asignarle algún valor específico, pero este valor lo asignamos nosotros y puede no ser el real. Esto ensucia el dataset cuando en realidad queremos limpiarlo. Por lo tanto, la mejor opción que podemos tomar es eliminar estas dos filas y no considerarlas en el análisis.

**Importante:** la eliminación de las filas que contienen valores nulos en al menos una columna es útil cuando esta cantidad no es considerable (en la mayoría de los casos, podemos hablar de aproximadamente 10% del total de registros del dataset).

In [19]:
df_titanic.dropna(subset=['Embarked'], inplace=True)

# Validamos la modificación
df_titanic.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64

In [20]:
registrar("Tratamiento de valores faltantes",
          "Embarked",
          "Eliminación de las filas",
          "Existen solamente 2 filas con nulos en la variable. Es una cantidad insignificante y la eliminación de estos registros no modificará el desarrollo del análisis")

Ahora nos queda solamente la edad.

In [21]:
proporcion_nulos = df_titanic['Age'].isna().sum()/df_titanic.shape[0]
print(f'La proporción de valores nulos es de {round(proporcion_nulos*100,2)}%')

La proporción de valores nulos es de 19.91%


En este caso, la proporción no es muy elevada (como en `Cabin`) pero no es tan baja (como en `Embarked`). Si queremos eliminar las filas con valores nulos estaríamos perdiende la quinta parte de todos los datos, una cantidad muy elevada de información. Por lo tanto, recurriremos a otras técnicas.

Una posibilidad es imputar los valores nulos con algún valor. Como no conocemos qué valores pudieron tener realmente, entonces le asignamos algún valor que no afecte al comportamiento en general. Las opciones son la media, si la variable tiene una distribución simétrica de datos; la mediana, si la variable tiene una distribución muy asímetrica; y la moda, si la variable es de tipo cualitativa.

La otra posibilidad es segmentar esta variable en distintas clases. De esta forma, quedarían categorías como "bebé", "adolescente", "adulto", "anciano" y "sin dato". Estaremos perdiendo precisión, pero mantenemos todos los registros en el análisis de datos.

Para este caso, decidiremos imputar los nulos de la variable `Age` por la media.

In [22]:
df_titanic['Age'].fillna(df_titanic['Age'].mean(), inplace=True)

# Validamos la modificación
df_titanic.isna().sum()

C:\Users\Pollo\AppData\Local\Temp\ipykernel_15344\4243601010.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_titanic['Age'].fillna(df_titanic['Age'].mean(), inplace=True)


Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      0
dtype: int64

In [23]:
registrar("Tratamiento de valores faltantes",
          "Age",
          "imputación por la media",
          "Casi el 20% de los valores son faltantes. Es poco para eliminar la columna y es mucho para eliminar las filas. Así que se decide imputar por la media, ya que presenta una distribución levemente simétrica")

### 4- Tratamiento de valores erroneos

Ahora hay que valor que los datos que tenemos ahora son válidos y correctos. Para eso, deberemos usar un poco de nuestra razón y sentido común para encontrar posibles errores. Rápidamente podemos notar que todas las variables, excepto `Fare`, contienen valores posibles.

In [24]:
for i in ['Survived','Pclass','Sex','SibSp','Parch','Embarked']:
    print(f'Los valores únicos de la variable {i} son: {df_titanic[i].unique()}')

Los valores únicos de la variable Survived son: [0 1]
Los valores únicos de la variable Pclass son: [3 1 2]
Los valores únicos de la variable Sex son: <StringArray>
['male', 'female']
Length: 2, dtype: str
Los valores únicos de la variable SibSp son: [1 0 3 4 2 5 8]
Los valores únicos de la variable Parch son: [0 1 2 5 3 4 6]
Los valores únicos de la variable Embarked son: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str


In [25]:
df_titanic['Age'].describe()

count    712.000000
mean      29.642093
std       14.492933
min        0.420000
25%       20.000000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

Observemos el caso de la variable `Fare`

In [26]:
df_titanic['Fare'].describe()

count    889.000000
mean      32.096681
std       49.697504
min        0.000000
25%        7.895800
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

Podemos notar dos valores sospechosos:

* Fare = 0. Esto significaría que el pasaje tuvo un costo de 0, lo cual no parece algo coherente. Incluso, este valor se relaciona con diferente valores de `Pclass`, lo cual tampoco tiene sentido. Por ende, consideraremos el valor 0 como un error.
* Fare = 512. Este valor, aunque es posible, es muy diferente de los demás. Encima, el segundo boleto más caro costó 263. Lo que puede aumentar más las sospechas. Sin embargo, como estamos hablando del titanic que tenía camarotes exclusivos de primera clase, podemos pensar que este valor pertenece a este tipo de boletos. Por lo tanto, lo consideraremos como un valor atípico y no como un error.



In [27]:
df_titanic[(df_titanic['Fare']==0)|(df_titanic['Fare']>500)]

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
179,0,3,male,36.0,0,0,0.0000,S
258,1,1,female,35.0,0,0,512.3292,C
263,0,1,male,40.0,0,0,0.0000,S
271,1,3,male,25.0,0,0,0.0000,S
277,0,2,male,NaN,0,0,0.0000,S
302,0,3,male,19.0,0,0,0.0000,S
413,0,2,male,NaN,0,0,0.0000,S
466,0,2,male,NaN,0,0,0.0000,S
481,0,2,male,NaN,0,0,0.0000,S
597,0,3,male,49.0,0,0,0.0000,S


Para sanar el error interpretado del Fare=0, eliminaremos las filas (ya que son pocas las filas afectadas)

In [28]:
df_titanic = df_titanic[~(df_titanic['Fare']==0)]

# Validamos la modificación imprimiendo el resultado de la expresión para ver los registros restantes
df_titanic['Fare'].describe()

count    874.000000
mean      32.647539
std       49.942710
min        4.012500
25%        7.925000
50%       14.500000
75%       31.275000
max      512.329200
Name: Fare, dtype: float64

In [29]:
registrar("Tratamiento de valores erroneos",
         "Fare",
          "Eliminación de filas con Fare == 0",
          "Se considera un Fare igual a 0 como un error. Se eliminan las filas por ser una proporción pequeña de registros")

### 5- Detección de valores atípicos

Por último, veamos si podemos considerar algunos valores como outliers. Recordamos que un outlier o valor atípico es aquel que se encuentra alejado del comportamiento del resto de los datos y, por consiguiente, se lo considera fuera de la norma o un caso extraño.

Pero guarda, estos casos no deben considerarse errores, ya que un valor fuera de lo común puede ocurrir y ser completamente válido. Por lo tanto, estos casos no deben ser tratados; simplemente daremos a conocer estas existencias para más adelante.

Para esto, utilizaremos una función que nos permita calcular un outlier mediante las dos siguientes fórmulas:


In [30]:
def detectar_outliers(serie, method='iqr'):
    if method=='iqr':
      q1 = serie.quantile(0.25)
      q3 = serie.quantile(0.75)
      iqr = q3 - q1
      limite_inferior = q1 - 1.5 * iqr
      limite_superior = q3 + 1.5 * iqr
    elif method=='3sig':
      media = serie.mean()
      desviacion_estandar = serie.std()
      limite_inferior = media - 3 * desviacion_estandar
      limite_superior = media + 3 * desviacion_estandar
    else:
      raise ValueError("El método debe ser 'iqr' o '3sig'.")

    outliers = serie[(serie < limite_inferior) | (serie > limite_superior)]
    return outliers

In [31]:
for i in ['Age','SibSp','Parch','Fare']:
    print(f'Cantidad de valores atípicos de la variable {i}: {detectar_outliers(df_titanic[i]).count()}')


Cantidad de valores atípicos de la variable Age: 8
Cantidad de valores atípicos de la variable SibSp: 46
Cantidad de valores atípicos de la variable Parch: 213
Cantidad de valores atípicos de la variable Fare: 114


In [32]:
# Visualización de los valores atípicos de una variable
detectar_outliers(df_titanic['Age']).sort_values()

33     66.0
672    70.0
745    70.0
116    70.5
493    71.0
96     71.0
851    74.0
630    80.0
Name: Age, dtype: float64

## Resumen de limpieza de datos

In [33]:
pd.DataFrame(registros_bitacora)

,problema_detectado,variable_afectada,decision_tomada,justificacion
0,Selección de variables para el análisis,PassengerId-Name-Ticket,Eliminación de las columnas,Estas columnas contienen nombres y códigos de ...
1,Tratamiento de valores faltantes,Cabin,Eliminación de la columna,"La columna contiene 77% de valores faltantes, ..."
2,Tratamiento de valores faltantes,Embarked,Eliminación de las filas,Existen solamente 2 filas con nulos en la vari...
3,Tratamiento de valores faltantes,Age,imputación por la media,Casi el 20% de los valores son faltantes. Es p...
4,Tratamiento de valores erroneos,Fare,Eliminación de filas con Fare == 0,Se considera un Fare igual a 0 como un error. ...


## Conclusión

Finalizamos así la limpieza del dataset. De esta manera, conseguimos una tabla sin problemas, lista para pasar por las próximas fases del análisis de datos.

Es importante realizar siempre la limpieza de datos al momento de recibir un conjunto de datos a estudiar. Los pasos presentados son los mínimos e indispensables para el análisis de datos. Existen muchas técnicas más para la limpieza de los datos, pero con esto será suficiente para avanzar con el Trabajo Práctico Integrador y con la materia.

# Práctica autónoma

Ahora vas a trabajar con un segundo conjunto de datos para que realices la limpieza correspondiente y construyas una bitácora propia.


## Dataset: Google Play Store Apps

Trabajaremos con el dataset 'Google Play Store Apps'. Este conjunto de datos contiene información sobre aplicaciones disponibles en Google Play Store, incluyendo categorías, calificaciones, reseñas, tamaños, número de instalaciones y más. Es útil para analizar tendencias del mercado de aplicaciones, la popularidad de ciertas categorías o el impacto de las características en las calificaciones de los usuarios.

Tu objetivo será limpiar todos los problemas que encuentres y dejarlo listo para un análisis profundo. Tené en cuenta los pasos hechos para limpiar el dataset titanic.

Aquí se describen en detalle las variables del dataset:

| Variable         | Descripción                                                                 |
| :--------------- | :-------------------------------------------------------------------------- |
| `App`            | Nombre de la aplicación.                                                    |
| `Category`       | Categoría a la que pertenece la aplicación (e.g., 'ART_AND_DESIGN', 'GAME'). |
| `Rating`         | Calificación promedio de la aplicación por los usuarios (1.0-5.0).           |
| `Reviews`        | Número de reseñas de usuarios para la aplicación.                            |
| `Size`           | Tamaño de la aplicación.                                                    |
| `Installs`       | Número de veces que la aplicación ha sido instalada.                         |
| `Type`           | Tipo de aplicación (e.g., 'Free', 'Paid').                                  |
| `Price`          | Precio de la aplicación (en USD, si es de pago).                             |
| `Content Rating` | Clasificación de contenido de la aplicación (e.g., 'Everyone', 'Teen').     |
| `Genres`         | Géneros a los que pertenece la aplicación.                                  |
| `Last Updated`   | Fecha de la última actualización de la aplicación.                           |
| `Current Ver`    | Versión actual de la aplicación.                                            |
| `Android Ver`    | Versión mínima de Android requerida para la aplicación.                     |


Con la siguiente celda podrás obtener el dataset completo. Si llega a generar errores o problemas, podés descargar el dataset que está subido en el aula virtual, subirlo a la carpeta "Archivos" de colab y ejecutar el siguiente código:

```
df_playstore = pd.load_csv('\content\google-play-store-apps.csv')
```



In [34]:
from pathlib import Path

DATA_DIR = Path('../datos')
df_playstore = pd.read_csv(DATA_DIR / 'google-play-store-apps.csv')
print('Dataset cargado:', df_playstore.shape)

Dataset cargado: (10841, 13)


In [35]:
df_playstore.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Diagnóstico inicial

Revisamos dimensiones, tipos de datos y estadísticos básicos antes de modificar el dataset.

In [36]:
print('Dimensiones:', df_playstore.shape)
df_playstore.info()

Dimensiones: (10841, 13)
<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


In [37]:
df_playstore.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
App,10841,9660,ROBLOX,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,10841,34,FAMILY,1972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rating,9367.0,NaN,NaN,NaN,4.193338,0.537431,1.0,4.0,4.3,4.5,19.0
Reviews,10841,6002,0,596,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Size,10841,462,Varies with device,1695,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Installs,10841,22,"1,000,000+",1579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Type,10840,3,Free,10039,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price,10841,93,0,10040,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Content Rating,10840,6,Everyone,8714,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Genres,10841,120,Tools,842,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
print('Valores nulos por columna:')
print(df_playstore.isna().sum().sort_values(ascending=False))

Valores nulos por columna:
Rating            1474
Current Ver          8
Android Ver          3
Content Rating       1
Type                 1
Size                 0
Reviews              0
Category             0
App                  0
Price                0
Installs             0
Last Updated         0
Genres               0
dtype: int64


In [39]:
print('Registros duplicados (todas las columnas):', df_playstore.duplicated().sum())
print('Duplicados App + Category:', df_playstore.duplicated(subset=['App', 'Category']).sum())

Registros duplicados (todas las columnas): 483
Duplicados App + Category: 1096


In [40]:
print('Rating fuera de rango 1-5:', (df_playstore['Rating'] > 5).sum())
print('Type inconsistentes:')
print(df_playstore['Type'].value_counts(dropna=False))

Rating fuera de rango 1-5: 1
Type inconsistentes:
Type
Free    10039
Paid      800
NaN         1
0           1
Name: count, dtype: int64


### Preparativos

Conservamos el dataset original y trabajamos sobre una copia. También iniciamos una bitácora propia para esta práctica autónoma.

In [41]:
df_playstore_original = df_playstore.copy()
df = df_playstore.copy()

registros_bitacora_ps = []

def registrar_ps(problema, variable, decision, justificacion):
    registros_bitacora_ps.append({
        'problema_detectado': problema,
        'variable': variable,
        'decision': decision,
        'justificacion': justificacion,
    })

### 1. Fila corrupta y duplicados

In [42]:
filas_corruptas = df['Rating'] > 5
print('Filas con Rating imposible (>5):', filas_corruptas.sum())
df.loc[filas_corruptas, ['App', 'Category', 'Rating', 'Reviews']]

Filas con Rating imposible (>5): 1


,App,Category,Rating,Reviews
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M


In [43]:
if filas_corruptas.any():
    df = df.loc[~filas_corruptas].copy()
    registrar_ps(
        'Fila con columnas desplazadas (Rating=19, Category=1.9)',
        'Todas',
        'Eliminar fila corrupta',
        'Es un error de registro, no una app válida; distorsiona tipos y estadísticos.',
    )

dup_total = df.duplicated().sum()
print('Duplicados exactos restantes:', dup_total)
if dup_total:
    df = df.drop_duplicates().copy()
    registrar_ps(
        'Registros idénticos en todas las columnas',
        'Todas',
        'Eliminar duplicados exactos (keep=first)',
        'Son copias literales del mismo registro, no aportan información.',
    )

dup_app_cat = df.duplicated(subset=['App', 'Category']).sum()
print('Duplicados App+Category:', dup_app_cat)
if dup_app_cat:
    df = df.drop_duplicates(subset=['App', 'Category'], keep='first').copy()
    registrar_ps(
        'Misma app repetida en la misma categoría',
        'App, Category',
        'Conservar primera aparición',
        'Indica republicación duplicada en la fuente; una fila por app-categoría.',
    )

print('Filas después de deduplicación:', len(df))

Duplicados exactos restantes: 483


Duplicados App+Category: 613
Filas después de deduplicación: 9744


### 2. Valores faltantes

In [44]:
prop_nulos_rating = df['Rating'].isna().mean()
print(f'Rating nulos: {df["Rating"].isna().sum()} ({prop_nulos_rating:.1%})')
print(f'Current Ver nulos: {df["Current Ver"].isna().sum()}')
print(f'Type nulos: {df["Type"].isna().sum()}')

Rating nulos: 1464 (15.0%)
Current Ver nulos: 8
Type nulos: 1


In [45]:
registrar_ps(
    'Calificaciones ausentes',
    'Rating',
    'Conservar NaN',
    'Imputar distorsionaría la variable objetivo de popularidad; se analizará con filtros.',
)
registrar_ps(
    'Versión actual ausente',
    'Current Ver',
    'Conservar NaN',
    'No es crítica para el análisis de ratings/instalaciones en esta etapa.',
)

### 3. Tipos incorrectos y categorías inconsistentes

In [46]:
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
registrar_ps(
    'Reviews almacenado como texto',
    'Reviews',
    'Convertir a numérico',
    'Permite calcular estadísticos y detectar atípicos.',
)

def parse_installs(valor):
    if pd.isna(valor):
        return pd.NA
    return int(str(valor).replace(',', '').replace('+', ''))

df['Installs_num'] = df['Installs'].map(parse_installs)
registrar_ps(
    'Installs con formato 10,000+',
    'Installs',
    'Crear Installs_num entero',
    'El signo + y las comas impiden operar; se conserva la columna original.',
)

df['Price_num'] = (
    df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .replace('0', 0)
)
df['Price_num'] = pd.to_numeric(df['Price_num'], errors='coerce')
registrar_ps(
    'Price con símbolo $',
    'Price',
    'Crear Price_num en USD',
    'Facilita comparar apps pagas y validar coherencia con Type.',
)

mask_type_invalido = df['Type'].isin(['0']) | df['Type'].isna()
print('Type inválido o nulo:', mask_type_invalido.sum())
df.loc[mask_type_invalido & (df['Price_num'] == 0), 'Type'] = 'Free'
df.loc[mask_type_invalido & (df['Price_num'] > 0), 'Type'] = 'Paid'
registrar_ps(
    "Type con valor '0' o nulo",
    'Type',
    'Corregir según Price_num',
    'Free/Paid debe ser coherente con el precio; no se eliminaron filas.',
)

df['Type'] = df['Type'].astype('category')

Type inválido o nulo: 1


### 4. Tamaño de la app (Size)

In [47]:
def parse_size_mb(valor):
    if pd.isna(valor) or valor == 'Varies with device':
        return pd.NA
    texto = str(valor).strip()
    if texto.endswith('k'):
        return float(texto[:-1]) / 1024
    if texto.endswith('M'):
        return float(texto[:-1])
    return pd.NA

df['Size_MB'] = df['Size'].map(parse_size_mb)
registrar_ps(
    'Size con unidades k/M y texto especial',
    'Size',
    'Crear Size_MB numérico; conservar original',
    "'Varies with device' no es un error sino información válida.",
)

### 5. Valores atípicos (IQR) en Rating y Reviews

In [48]:
def detectar_outliers_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < lim_inf) | (s > lim_sup)]

out_rating = detectar_outliers_iqr(df['Rating'])
out_reviews = detectar_outliers_iqr(df['Reviews'])
print('Atípicos Rating (IQR):', len(out_rating))
print('Atípicos Reviews (IQR):', len(out_reviews))
print('Rating válido min/max:', df['Rating'].min(), df['Rating'].max())

Atípicos Rating (IQR): 492
Atípicos Reviews (IQR): 1677
Rating válido min/max: 1.0 5.0


In [49]:
registrar_ps(
    'Reviews con valores extremos (IQR)',
    'Reviews',
    'Conservar; marcar para análisis',
    'Apps muy populares pueden tener millones de reseñas; no son errores.',
)
registrar_ps(
    'Rating dentro de 1-5 tras limpieza',
    'Rating',
    'Conservar distribución',
    'Tras eliminar la fila corrupta, no hay ratings imposibles.',
)

### Dataset limpio y bitácora

In [50]:
df_playstore_limpio = df.copy()
print('Shape final:', df_playstore_limpio.shape)
df_playstore_limpio.head()

Shape final: (9744, 16)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_num,Price_num,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,10000,0.0,19.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000,0.0,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,5000000,0.0,8.7
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,50000000,0.0,25.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,100000,0.0,2.8


In [51]:
bitacora_ps = pd.DataFrame(registros_bitacora_ps)
bitacora_ps

,problema_detectado,variable,decision,justificacion
0,"Fila con columnas desplazadas (Rating=19, Cate...",Todas,Eliminar fila corrupta,"Es un error de registro, no una app válida; di..."
1,Registros idénticos en todas las columnas,Todas,Eliminar duplicados exactos (keep=first),"Son copias literales del mismo registro, no ap..."
2,Misma app repetida en la misma categoría,"App, Category",Conservar primera aparición,Indica republicación duplicada en la fuente; u...
3,Calificaciones ausentes,Rating,Conservar NaN,Imputar distorsionaría la variable objetivo de...
4,Versión actual ausente,Current Ver,Conservar NaN,No es crítica para el análisis de ratings/inst...
5,Reviews almacenado como texto,Reviews,Convertir a numérico,Permite calcular estadísticos y detectar atípi...
6,"Installs con formato 10,000+",Installs,Crear Installs_num entero,El signo + y las comas impiden operar; se cons...
7,Price con símbolo $,Price,Crear Price_num en USD,Facilita comparar apps pagas y validar coheren...
8,Type con valor '0' o nulo,Type,Corregir según Price_num,Free/Paid debe ser coherente con el precio; no...
9,Size con unidades k/M y texto especial,Size,Crear Size_MB numérico; conservar original,'Varies with device' no es un error sino infor...


## Conclusión – Práctica autónoma

El dataset Google Play Store presentaba **problemas de calidad reales**: una fila corrupta por desplazamiento de columnas, cientos de duplicados, variables numéricas en formato texto (`Reviews`, `Installs`, `Price`, `Size`) y categorías inconsistentes en `Type`. Se conservó el original (`df_playstore_original`) y se trabajó sobre copias.

Las decisiones priorizaron **no eliminar datos solo por reglas automáticas**: los atípicos de `Reviews` se conservaron por ser apps legítimamente populares, y los `Rating` faltantes se mantuvieron para no sesgar el análisis. El dataset quedó listo para análisis exploratorio con columnas derivadas (`Installs_num`, `Price_num`, `Size_MB`).


## Consignas

1. Creá una copia de trabajo y mantené intacto el dataset original.
2. Revisá dimensiones de dataset, tipos de varaiables y estadísticos básicos.
3. Realizá todos los pasos para la limpieza de datos. Recordá siempre mantener un registro de todas las modificaciones hechas en el dataset.
4. Finalizá con una breve conclusión de cómo viste el dataset, qué cosas encontraste, qué modificaciones tuviste que realizar y por qué.

Al finaìzar, revisá los siguientes puntos para corroborar que todo estuvo ok.

- [ ] Trabajé sobre una copia.
- [ ] Diferencié faltantes, inconsistencias, duplicados y posibles valores atípicos.
- [ ] No eliminé registros únicamente porque una regla los señaló.
- [ ] Justifiqué cada cambio.
- [ ] Completé la bitácora.
- [ ] El notebook puede ejecutarse desde el comienzo.